In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q \
langchain \
langchain-community \
langchain-core \
langchain-google-genai \
langchain-huggingface \
sentence-transformers \
faiss-cpu

In [3]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

/tmp/ipykernel_24298/1729546305.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
from getpass import getpass
import os

os.environ["NVIDIA_API_KEY"] = getpass("Enter your NVIDIA API Key: ")

Enter your NVIDIA API Key: ··········


In [ ]:
print("NVIDIA_API_KEY is set:", "NVIDIA_API_KEY" in os.environ)


In [6]:
!pip install -qU langchain-nvidia-ai-endpoints

In [7]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

print("NVIDIA Chat Imported Successfully")

NVIDIA Chat Imported Successfully


In [8]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
    temperature=0
)

print("llm Loaded Successfully!")

llm Loaded Successfully!


In [9]:
response = llm.invoke("Say hello.")

print(response.content)

Hello.


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device":"cpu"},
    encode_kwargs={"normalize_embeddings":True}
)

vector_db = FAISS.load_local(
    "/content/drive/MyDrive/LegalResearch/vectorstore",
    embedding_model,
    allow_dangerous_deserialization=True
)

print(vector_db.index.ntotal)

In [ ]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

print("Retriever created")

In [ ]:
question = "What are the grounds for divorce under the Hindu Marriage Act?"

docs = retriever.invoke(question)

print(len(docs))

In [ ]:
context = "\n\n".join(
    doc.page_content
    for doc in docs
)

In [ ]:
sources = "\n".join(
    f"- {doc.metadata.get('source')} (Page {doc.metadata.get('page',0)+1})"
    for doc in docs
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an expert Indian Legal Research Assistant.

Your task is to answer questions ONLY using the retrieved legal documents.

Rules:
- Do NOT make up laws, sections, articles, or judgments.
- Do NOT use outside knowledge.
- If the answer is not available in the retrieved context, clearly say:
  "I could not find sufficient information in the provided legal documents."
- If relevant case laws are present in the retrieved context, include them.
- If no case law is present, write:
  "No related case law found in the retrieved documents."

Return your answer in the following format:

## Answer
<Answer>

## Relevant Act
<Name of the Act>

## Relevant Section / Article
<Section or Article Number>

## Related Landmark Cases
<List case names if available>

## Explanation
<A short explanation in 3-5 sentences>

Context:
{context}

Question:
{question}
""")

In [ ]:
chain = prompt | llm

In [ ]:
question = "What are the grounds for divorce under the Hindu Marriage Act?"

docs = retriever.invoke(question)

print(len(docs))

In [ ]:
docs = retriever.invoke(question)

In [ ]:
context = "\n\n".join(
    doc.page_content
    for doc in docs
)

In [ ]:
response = chain.invoke({
    "context": context,
    "question": question
})

In [ ]:
response = llm.invoke("What is Article 21? Answer in one sentence.")
print(response.content)
